# 08 · Scale the analysis lazily

## Context

A monthly cube has grown beyond comfortable in-memory analysis. The scientific
method should remain readable even when execution must be divided into bounded
chunks.

## Question

Can we compose an anomaly-variance analysis without loading the source cube,
then compute only the small final map needed for interpretation?

## Analysis story

We will place deterministic values behind Dask, apply the same two-verb grammar
used for in-memory cubes, inspect the deferred graph, and materialize only at an
explicit execution boundary.

In [ ]:
import dask.array as da
import numpy as np
import pandas as pd
import xarray as xr

from cubedynamics import pipe, verbs as v

# Start with ordinary NumPy values, then wrap them in a Dask array split into
# bounded chunks: 6 time steps × 4 rows × 5 columns per chunk.
rng = np.random.default_rng(7)
values = rng.normal(size=(24, 8, 10)).astype("float32")
lazy_values = da.from_array(values, chunks=(6, 4, 5))

# xarray adds scientific coordinates without changing the lazy Dask backing.
cube = xr.DataArray(
    lazy_values,
    dims=("time", "y", "x"),
    coords={
        "time": pd.date_range("2023-01-01", periods=24, freq="MS"),
        "y": np.arange(8),
        "x": np.arange(10),
    },
    name="signal",
    attrs={"source": "deterministic synthetic vignette"},
)

assert cube.chunks is not None
cube

## Pipe · Describe the work without executing it

Nothing about the analytical sentence mentions Dask. Laziness is a property of
the data, and the verbs preserve it.

In [ ]:
result = (
    pipe(cube)
    | v.anomaly(dim="time")
    | v.variance(dim="time", keep_dim=False)
).unwrap()

# chunks proves the arrays remain lazy. Counting graph tasks is safe because it
# inspects the plan rather than computing the array values.
assert result.chunks is not None
graph_tasks = len(result.data.__dask_graph__())
graph_tasks

## Figure · Compute only at the interpretation boundary

`compute()` appears once and visibly. It materializes the reduced 2D result,
not the full source cube.

In [ ]:
import matplotlib.pyplot as plt

# compute() is the intentional execution boundary. Only the small final map is
# materialized, and that map—not the full source cube—is sent to Matplotlib.
materialized = result.compute()
assert materialized.chunks is None
fig, ax = plt.subplots(figsize=(7, 4.5), constrained_layout=True)
materialized.plot(ax=ax, cmap="magma", cbar_kwargs={"label": "anomaly variance"})
ax.set_title(f"Computed final map · graph previously had {graph_tasks} tasks")
plt.show()

## What the figure tells us

The map locates pixels with the greatest variance after removing each pixel's
mean. More importantly, the method stayed identical to an in-memory pipe; only
the final, explicit execution boundary changed.

## Try the next variation

Change the chunk sizes and inspect `graph_tasks` again. The execution plan will
change while the scientific pipe remains the same.